<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/04_multiphase_and_fields.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 4: Visualizing the Invisible -- Multi-Phase Bottlenecks

**The Pain Point:** A battery fails at high charge rates, but standard binary (solid/pore) simulations say transport is fine. The missing ingredient? The **Carbon-Binder Domain (CBD)** -- a third phase that partially blocks ion flow. Without multi-phase capability, researchers are blind to these bottlenecks.

**The Solution:** OpenImpala's advanced solvers handle spatially varying transport coefficients $D(x,y,z)$ to homogenize multi-phase composites. We can extract the solved potential field back into Python to plot the exact locations of lithium-ion bottlenecks.

**In this tutorial we will:**
1. Create a 3-phase microstructure (Matrix, Inclusions, and Pores).
2. Use AMReX's parameter database (`ParmParse`) to assign specific conductivities to each phase.
3. Solve the Effective Diffusivity cell problem using `openimpala.core`.
4. Extract the solved corrector field ($\chi$) to visualize the flux pathways in Matplotlib.

In [ ]:
# Install OpenImpala and visualization libraries
!pip install -q openimpala matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import openimpala as oi
import openimpala.core as oic

# We also import pyAMReX to interact directly with the C++ data structures
import amrex.space3d as amrex

print(f"OpenImpala version {oi.__version__} loaded.")

## 1. Creating a 3-Phase Composite

Let's create a domain with three phases:
- **Phase 0 (Solid Matrix):** Completely insulating ($D = 0.0$)
- **Phase 1 (Electrolyte/Pore):** Highly conductive ($D = 1.0$)
- **Phase 2 (Binder/Carbon):** Partially conductive ($D = 0.1$)

In [ ]:
N = 64
micro = np.zeros((N, N, N), dtype=np.int32)  # Phase 0 everywhere

# Add a central "pore" channel (Phase 1)
micro[16:48, 16:48, :] = 1

# Add some "binder" blockages inside the pore (Phase 2)
micro[24:40, 24:40, 10:20] = 2
micro[20:44, 20:44, 40:50] = 2

plt.figure(figsize=(5, 5))
plt.imshow(micro[:, N//2, :], cmap='viridis')
plt.title("Slice of 3-Phase Composite (Y-Z plane)")
plt.xlabel("Z (Flow Direction)")
plt.ylabel("X")
# 0=Purple (Solid), 1=Teal (Pore), 2=Yellow (Binder)
plt.show()

## 2. Configuring Multi-Phase Physics & 3. Extracting the Solution Field

When dealing with more than a binary threshold, we need to pass configuration
directly to the underlying C++ AMReX engine. We do this using `amrex.ParmParse`
*inside* the OpenImpala Session.

> **Note on Advanced APIs:** We are exposing internal AMReX classes here (like `ParmParse` and `MultiFab`) as well as OpenImpala facade helpers. While powerful, these are low-level tools intended for users comfortable with C++ HPC data structures.

Once the solver finishes, we will extract the computed corrector field ($\chi_z$)
from the distributed AMReX `MultiFab` back into a standard NumPy array.
Because AMReX objects require an active context, both the solving and
extraction steps must happen within the same `with oi.Session():` block.

In [ ]:
with oi.Session():
    # ==========================================
    # PART 1: Configure and Solve
    # ==========================================
    # 1. Define Multi-Phase settings in the AMReX global database
    # This tells the C++ backend: Phase 1 has D=1.0, Phase 2 has D=0.1
    pp = amrex.ParmParse("tortuosity")
    pp.addarr("active_phases", [1, 2])
    pp.addarr("phase_diffusivities", [1.0, 0.1])

    # 2. Convert NumPy array to AMReX iMultiFab
    # We use the facade helper to handle the domain decomposition
    geom, ba, dm, mf = oi.facade._numpy_to_imultifab(micro, max_grid_size=32)

    # 3. Initialize the Advanced Homogenization Solver
    # We tell it we are solving in the Z-direction (Direction.Z)
    print("Initializing Effective Diffusivity Solver...")
    solver = oic.EffectiveDiffusivityHypre(
        geom, ba, dm, mf,
        phase_id=1,                  # Primary reference phase
        dir=oic.Direction.Z,         # Flow direction
        solver_type=oic.EffDiffSolverType.FlexGMRES,
        results_path=".",
        verbose=1,
        write_plotfile=False
    )

    # 4. Run the solver
    print("Solving PDEs via HYPRE...")
    converged = solver.solve()
    print(f"Solver converged: {converged} in {solver.iterations} iterations.")

    # ==========================================
    # PART 2: Extracting the PDE Solution Field
    # ==========================================
    # Create an empty AMReX floating-point MultiFab to hold the result
    chi_field = amrex.MultiFab(ba, dm, 1, 1)

    # Ask the C++ solver to copy the HYPRE solution into our MultiFab
    solver.get_chi_solution(chi_field)

    # Convert the distributed AMReX MultiFab back into a single NumPy array
    # (Note: In a massive distributed MPI run, you wouldn't gather this to one node.
    # You would use `write_plotfile=True` and view it in ParaView/yt instead).
    chi_numpy = np.zeros((N, N, N), dtype=np.float64)

    for mfi in chi_field:
        # Get the valid box (no ghost cells) for this chunk
        bx = mfi.validbox()
        lo = bx.small_end
        hi = bx.big_end

        # Extract the local chunk as a 3D NumPy array
        local_arr = chi_field.array(mfi).to_numpy()

        # Place it into our global array
        chi_numpy[lo[0]:hi[0]+1, lo[1]:hi[1]+1, lo[2]:hi[2]+1] = local_arr[:,:,:,0]

## 4. Visualizing the Bottlenecks

Now that we have the raw PDE solution in NumPy, we can plot the potential field.
Notice how the gradient (flux) changes as it is forced to move through the
resistive "binder" blockages.

In [ ]:
# Plot the potential field slice
plt.figure(figsize=(8, 6))

# Mask out the solid phase (Phase 0) so it shows up as white/empty
masked_chi = np.ma.masked_where(micro[:, N//2, :] == 0, chi_numpy[:, N//2, :])

img = plt.imshow(masked_chi, cmap='magma')
plt.colorbar(img, label="Corrector Potential ($\\chi_z$)")
plt.title("Internal Potential Field (Flow in Z-direction)")
plt.xlabel("Z (Flow Direction)")
plt.ylabel("X")
plt.axis('off')
plt.show()

## What's Next?

You just solved the effective diffusivity cell problem on a multi-phase composite and visualized the internal potential field. The steep gradients around the binder blockages show exactly where transport resistance concentrates.

**Continue the journey:**
- [Tutorial 5: AI Data Factory](05_surrogate_modelling.ipynb) -- Generate thousands of microstructures to train ML surrogates.
- [Tutorial 6: Topology Optimisation](06_topology_optimisation.ipynb) -- Use OpenImpala inside a generative design loop.
- [Tutorial 7: Laptop to Supercomputer](07_hpc_scaling.ipynb) -- Scale up to massive datasets with MPI.